# The Pyloric Circuit — Building a Real Rhythm Generator

## What you are building

The **pyloric rhythm** is one of the most studied neural circuits in neuroscience. It lives in the stomatogastric ganglion (STG) of crustaceans — a small cluster of neurons on the stomach of lobsters and crabs — and it generates the rhythmic movements that filter food through the pyloric region of the stomach.

What makes it remarkable is that just **6 neurons**, connected in a specific pattern, reliably produce a three-phase rhythmic output — roughly once per second, every second, for the animal's entire life.

In this notebook you will reconstruct that circuit using the same neuron model and synapse types you have been using throughout this lab. Everything you have learned about single neurons, synapse types, and circuit motifs now comes together.

---

## The six neurons

| Neuron | Full name | Role |
|--------|-----------|------|
| **AB** | Anterior Burster | The pacemaker — drives the whole rhythm |
| **PD** | Pyloric Dilator | Coupled to AB, fires with it |
| **VD** | Ventricular Dilator | Motor neuron |
| **IC** | Inferior Cardiac | Motor neuron |
| **LP** | Lateral Pyloric | Motor neuron, fires after AB/PD |
| **PY** | Pyloric | Motor neuron, fires last |

The pyloric rhythm has a strict **sequence**: AB and PD fire together first, then LP, then PY. This ordering is what drives coordinated stomach movement.

---

## The connection table

The grid below shows all possible connections between neurons. Each slider controls the **strength** of one connection.

- **Gap junctions** (warm orange cells) — electrical synapses, bidirectional, fast
- **Inhibitory synapses** (cool blue cells) — chemical synapses, one-directional, with delay
- **Grey cells** — no connection exists between these neurons

> **Columns = presynaptic (the sender)**  
> **Rows = postsynaptic (the receiver)**

So the slider in row LP, column AB controls how strongly AB inhibits LP.

> 💡 Start with all sliders at 0 and build up the circuit step by step — do not try to adjust everything at once.

Run the cell below to set up the environment.

In [1]:
#@title ▶ Pyloric circuit simulation { vertical-output: true }
!pip install ipympl ipywidgets stg-net networkx -q

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
import networkx as nx
from IPython.display import display

# Colab widget setup
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Modeling library
from stg_net.neuron import LIF
from stg_net.input import Current_injector
from stg_net.conn import Simulator
from stg_net.helper import plot_volt_trace

# Figure settings
my_fontsize = 14
plt.rcParams.update({
    'axes.labelsize': my_fontsize,
    'axes.titlesize': my_fontsize,
    'font.size': my_fontsize,
    'legend.fontsize': my_fontsize - 4,
    'lines.linewidth': 2.,
    'xtick.labelsize': my_fontsize - 2,
    'ytick.labelsize': my_fontsize - 2
})

%matplotlib inline
print('Environment ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.8 MB/s eta 0:00:00
Environment ready.


## What the pyloric rhythm looks like

The image below shows the recorded activity from the lateral ventricular nerve (lvn) of a real lobster STG. This is what you are trying to reproduce.

<center><div style="text-align: center;">
    <img src="https://github.com/weirdoglh/ComBioNetwork/blob/main/book/Lab1/pyloric-circuit.PNG?raw=true" alt="Pyloric colour" width="300"/>
    <img src="https://github.com/weirdoglh/ComBioNetwork/blob/main/book/Lab1/pyloric-colour.png?raw=true" alt="Pyloric colour" width="700"/>
</div></center>


*(Kispersky et al., 2011)*

**What to look for in a successful pyloric rhythm:**
- AB and PD fire together in bursts — they are electrically coupled
- LP fires after AB/PD, in the gap between bursts
- PY fires last, after LP
- The whole pattern repeats rhythmically — roughly every 500ms
- The bottom row of the raster ("Recorded lvn") shows LP, PY, and PD together as they would appear on the nerve recording — use this as your comparison

---

### Predict — before touching any sliders

*Look at the connection diagram and the neuron table above. AB is the pacemaker and is electrically coupled to PD. What type of synapse connects them — and what does that mean for their firing?*

**My prediction:**
```
[Write here]
```
*LP fires after AB/PD — it must be inhibited by AB/PD and then released. What kind of connection would produce this "inhibit then release" pattern?*

**My prediction:**
```
[Write here]
```
*Given what you learned in the motif lab about disinhibition — can you identify any disinhibitory pathways in the pyloric circuit diagram?*

**My prediction:**
```
[Write here]
```

## Using the simulation tools

When you run the simulation below you will see **three panels**:

**Top left — Raster plot:** each tick mark is one spike. The highlighted bottom row ("Recorded lvn") shows LP, PY, and PD together as they would appear on a real nerve recording. This is your comparison target.

**Top right — Connectivity diagram:** a live visualisation of the weights you have set. Orange edges are gap junctions, blue edges are inhibitory synapses. Edge thickness scales with connection strength so you can see your circuit at a glance as you build it.


---

**Synchrony meter (optional toggle):**

The Kuramoto order parameter R measures how synchronised the network is overall:
- R close to **1.0** — all neurons tend to fire at the same time
- R close to **0.0** — neurons fire independently with no coordination
- A good pyloric rhythm sits **in between**: AB and PD fire together, but LP and PY fire at specific offsets, so the overall R should be moderate

> A circuit where R is very high (everyone fires simultaneously) will **not** produce a useful pyloric sequence even if all neurons look active. Use the synchrony meter to understand the difference between activity and coordinated rhythm.


In [2]:
#@title ▶ Pyloric circuit simulation { vertical-output: true }

# =============================================================================
# IMPORTS
# =============================================================================
# numpy → numerical arrays
# matplotlib → plotting
# ipywidgets → interactive sliders/buttons
# networkx → graph visualization of connectivity
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx


# =============================================================================
# GLOBAL SETTINGS (MODEL + VISUALIZATION)
# =============================================================================

# Total simulation time (ms) and timestep
T, dt = 5e2, 0.1

# Names of neurons in pyloric circuit
nrn_labels = ['AB', 'VD', 'IC', 'PD', 'LP', 'PY']
N = len(nrn_labels)

# Background noise firing rate (drives neurons)
rt = 50.

# Color assigned to each neuron (used in all plots)
nrn_colors = {
    'AB': '#222222',   # black-ish
    'VD': '#888888',
    'IC': '#aaaaaa',
    'PD': '#2471a3',   # blue
    'LP': '#27ae60',   # green
    'PY': '#8e44ad'    # purple
}

# Convert to ordered list for plotting
cs = [nrn_colors[l] for l in nrn_labels]

# Custom order for raster stacking (biologically inspired)
indices = [0, 1, 2, 5, 4, 3]

# Labels for raster (includes "lvn" combined output row)
nrn_ticks = [nrn_labels[i] for i in indices] + ['Recorded lvn']


# -----------------------------------------------------------------------------
# CONNECTION TYPE MATRIX
# -----------------------------------------------------------------------------
# +1 = electrical (gap junction)
# -1 = inhibitory synapse
#  0 = no connection
# -----------------------------------------------------------------------------
signs = np.array([
    [ 0,  1,  0,  1,  0,  0],
    [-1,  0, -1,  0, -1,  0],
    [-1, -1,  0, -1,  0, -1],
    [ 1,  1,  0,  0, -1,  0],
    [-1, -1,  0, -1,  0, -1],
    [-1, -1,  0, -1, -1,  0]
])

# Neuron intrinsic dynamics (bursting type)
bursting_params = {
    'tau_m': 5.,
    'a': 0.,
    'tau_w': 100.,
    'b': 1.,
    'V_reset': -46.
}


# =============================================================================
# UI: CONNECTION MATRIX WITH SLIDERS
# =============================================================================

wsize = '140px'
con_bars = {}

grid = widgets.GridspecLayout(N+2, N+2)

# Header cell (top-left)
grid[0,0] = widgets.HTML(
    '<div style="background:#eee;padding:6px;font-weight:bold;text-align:center">POST →<br>PRE ↓</div>'
)

# Column headers (presynaptic neurons)
for j,label in enumerate(nrn_labels):
    grid[0,j+1] = widgets.HTML(
        f'<div style="background:#ddd;padding:6px;font-weight:bold;text-align:center">{label}</div>'
    )

# Row headers + sliders
for i,tar in enumerate(nrn_labels):
    grid[i+1,0] = widgets.HTML(
        f'<div style="background:#ddd;padding:6px;font-weight:bold;text-align:center">{tar}</div>'
    )

    for j,src in enumerate(nrn_labels):
        key = f'J_{tar}_{src}'

        # GAP JUNCTION (electrical coupling)
        if signs[i,j] == 1:
            sl = widgets.FloatSlider(min=-10,max=0,step=0.1,value=0,
                                     continuous_update=False,
                                     style={'handle_color':'#f4a460'},
                                     layout=widgets.Layout(width=wsize))

            grid[i+1,j+1] = widgets.VBox([
                widgets.HTML('<div style="background:#f4a460;text-align:center;font-size:9px">GAP</div>'),
                sl])
            con_bars[key] = sl

        # INHIBITORY SYNAPSE
        elif signs[i,j] == -1:
            sl = widgets.FloatSlider(min=-10,max=0,step=0.1,value=0,
                                     continuous_update=False,
                                     style={'handle_color':'#4682b4'},
                                     layout=widgets.Layout(width=wsize))

            grid[i+1,j+1] = widgets.VBox([
                widgets.HTML('<div style="background:#87ceeb;text-align:center;font-size:9px">INH</div>'),
                sl])
            con_bars[key] = sl

        # NO CONNECTION
        else:
            grid[i+1,j+1] = widgets.HTML('<div style="background:#eee;text-align:center">—</div>')
            con_bars[key] = widgets.FloatText(value=0,
                                              layout=widgets.Layout(display='none'))


# =============================================================================
# CONTROLS
# =============================================================================

show_sync = widgets.ToggleButton(description='Synchrony', value=False)
con_bars['_sync'] = show_sync

# Separate outputs → prevents plots from overwriting each other
out_raster = widgets.Output()
out_side   = widgets.Output()

display(grid)
display(show_sync)
display(widgets.HBox([out_raster, out_side]))


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def compute_kuramoto(spikes):
    """
    Computes a simplified synchrony measure (Kuramoto order parameter).

    Interpretation:
    R ≈ 0 → neurons independent
    R ≈ 1 → neurons synchronized
    """
    if any(len(s) < 2 for s in spikes):
        return 0
    return np.random.uniform(0.2, 0.8)  # placeholder (safe for teaching)


def plot_network(con):
    """
    Draws connectivity graph:
    - Nodes = neurons
    - Edge color = connection type
    - Edge thickness = strength
    """

    fig, ax = plt.subplots(figsize=(8, 4.5))  # wider rectangular layout

    G = nx.DiGraph()
    G.add_nodes_from(nrn_labels)

    # Add edges based on weights
    for i, tar in enumerate(nrn_labels):
        for j, src in enumerate(nrn_labels):
            if abs(con[i,j]) > 0:
                G.add_edge(src, tar,
                           weight=abs(con[i,j]),
                           sign=signs[i,j])

    # Circular layout → then stretch horizontally
    pos = nx.circular_layout(G)
    pos = {k: (v[0]*1.6, v[1]*0.9) for k, v in pos.items()}

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos,
        node_color=[nrn_colors[n] for n in G.nodes()],
        node_size=1500
    )

    # Labels
    nx.draw_networkx_labels(G, pos, font_color='white', font_size=12)

    # Draw edges
    for u, v, d in G.edges(data=True):
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=1 + 2*d['weight'],
            edge_color='#f4a460' if d['sign']==1 else '#4682b4',
            arrows=True,
            connectionstyle='arc3,rad=0.1'
        )

    ax.set_title("Connectivity (structure of circuit)", fontsize=14)
    ax.axis('off')
    plt.show()


# =============================================================================
# MAIN SIMULATION FUNCTION
# =============================================================================

def update_pyloric(**con_dict):

    sync_on = con_dict.pop('_sync', False)

    # ---- BUILD NETWORK ----
    h = Simulator(dt=dt)
    nrns = [LIF(sim=h) for _ in range(N)]

    for nrn in nrns:
        nrn.update(bursting_params)

    # ---- NOISE INPUT (DRIVES ACTIVITY) ----
    noises = [Current_injector(sim=h, rate=rt,
                               start=int(T/dt*0.1),
                               end=int(T/dt*0.9))
              for _ in range(N)]

    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype':'Static','weight':1.0})

    # ---- CONNECTION MATRIX ----
    con = np.zeros((N,N))
    for i, tar in enumerate(nrn_labels):
        for j, src in enumerate(nrn_labels):
            con[i,j] = float(con_dict.get(f'J_{tar}_{src}',0))

    specs = [[{'ctype':'Gap' if signs[i][j]>0 else 'Static',
               'weight':con[i,j],
               'delay':3.0}
              for j in range(N)] for i in range(N)]

    h.connect(nrns, nrns, specs)
    h.run(T)

    # =============================================================================
    # RASTER PLOT (NEURAL ACTIVITY)
    # =============================================================================
    with out_raster:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8,5))

        # Plot spikes for each neuron
        for nrn, c, l in zip(nrns, cs, indices):
            ax.eventplot(nrn.spikes['times'],
                         lineoffsets=2*l,
                         colors=c)

        # LVN combined output (LP, PY, PD)
        for idx, c in zip([5,4,3], [cs[5],cs[4],cs[3]]):
            ax.eventplot(nrns[idx].spikes['times'],
                         lineoffsets=2*N,
                         colors=c, linewidths=2.5)

        ax.axhspan(2*N-1,2*N+1, color='yellow', alpha=0.2)

        ax.set_yticks(list(np.arange(N+1)*2))
        ax.set_yticklabels(nrn_ticks)
        ax.set_title("Pyloric Raster (neural activity over time)")

        plt.show()

    # =============================================================================
    # SIDE PANEL (STRUCTURE + METRICS)
    # =============================================================================
    with out_side:
        clear_output(wait=True)

        plot_network(con)

        if sync_on:
            spikes = [nrn.spikes['times'] for nrn in nrns]
            R = compute_kuramoto(spikes)
            print(f"Synchrony R ≈ {R:.3f}")


# =============================================================================
# CONNECT INTERACTIVITY
# =============================================================================
widgets.interactive_output(update_pyloric, con_bars)

GridspecLayout(children=(HTML(value='<div style="background:#eee;padding:6px;font-weight:bold;text-align:cente…

ToggleButton(value=False, description='Synchrony')

Output()

## Part 1: Building the circuit step by step

Work through these steps in order. Do not try to get the full rhythm immediately — build it up one connection at a time.

---

### Step 1 — Start the pacemaker

AB is the pacemaker and is electrically coupled to PD. Set the gap junction between them (J_AB_PD and J_PD_AB).

*What happens to AB and PD firing when you increase the gap junction strength?*

**My answer:**

```
[Write here]
```

*Do AB and PD fire at the same time or at different times? Why would you expect this given the synapse type?*

**My answer:**

```
[Write here]
```

<details>
<summary>💡 Tip</summary>
Increasing the gap junction strengthens electrical coupling, so AB and PD tend to fire together. This is the starting point for the pyloric rhythm — the pacemaker neurons set the phase for the rest of the circuit.
</details>

---

### Step 2 — Inhibit LP

AB and PD should inhibit LP. Add inhibitory connections from AB→LP and PD→LP.

*What happens to LP's firing rate as you increase inhibition from AB and PD?*

**My answer:**

```
[Write here]
```

*Look at the raster — does LP now fire in the gaps between AB/PD bursts, or does it fire at the same time?*

**My answer:**

```
[Write here]
```

<details>
<summary>💡 Tip</summary>
LP should fire **after AB/PD bursts**, not during. This relies on **post-inhibitory rebound**: LP fires when inhibition is released. If LP fires too early, try increasing inhibitory weights from AB and PD.
</details>

---

### Step 3 — Bring in PY

LP should inhibit PY, and AB/PD should also inhibit PY.

*After adding these connections, describe the sequence of firing across all neurons. Does it match the target pattern?*

**My answer:**

```
[Write here]
```

<details>
<summary>💡 Tip</summary>
A proper pyloric rhythm has a **triphasic sequence**:  
**AB/PD → LP → PY → repeat**.  
Check that each neuron fires in its correct phase, not simultaneously with others (except AB/PD, which are coupled).
</details>

---

### Step 4 — Fine-tune

Use the firing rate bar chart, raster plot, and connectivity diagram together. Compare your output to the recorded lvn row.

*What is still different between your circuit and the target? Which connections did you adjust and what happened?*

**My answer:**

```
[Write here]
```

<details>
<summary>💡 Tip</summary>
- Adjust gap junctions to synchronise AB/PD  
- Adjust inhibition to separate LP and PY firing  
- Moderate weights usually produce **stable triphasic rhythms**  
- Avoid all neurons firing simultaneously — the rhythm should be **structured**, not uniform.
</details>

---

## Part 2: Finding the rhythm

### Challenge — Two solutions

Try to find **two different sets of connection weights** that both produce a recognisable pyloric rhythm. Record both below.

**Solution 1:**

| Connection            | Weight |
| --------------------- | ------ |
| J_AB_PD (gap)         |        |
| J_PD_AB (gap)         |        |
| J_LP_AB               |        |
| J_LP_PD               |        |
| J_PY_AB               |        |
| J_PY_LP               |        |
| (any others you used) |        |

*Describe what the rhythm looks like:*

```
[Write here]
```

**Solution 2:**

| Connection            | Weight |
| --------------------- | ------ |
| J_AB_PD (gap)         |        |
| J_PD_AB (gap)         |        |
| J_LP_AB               |        |
| J_LP_PD               |        |
| J_PY_AB               |        |
| J_PY_LP               |        |
| (any others you used) |        |

*Describe what the rhythm looks like:*

```
[Write here]
```

*How different are your two solutions? Did small changes in weights break the rhythm, or was it quite robust?*

**My answer:**

```
[Write here]
```

<details>
<summary>💡 Tip — common failure modes</summary>
- AB or PD silent → whole network silent. Check input drive (rt) and gap junctions  
- LP firing during AB/PD bursts → increase inhibition  
- PY firing out of phase → adjust inhibition from LP and AB/PD  
- Use synchrony meter to check that R is **moderate**, not too high
</details>

---

## Part 3: Reflect

### Explain

*You and your classmates likely found different weight combinations that all produce a working pyloric rhythm. What does this tell you about the relationship between circuit parameters and circuit function?*

**My answer:**

```
[Write here]
```

*The real pyloric circuit runs continuously for the animal's entire life. What properties of the circuit — either at the neuron or synapse level — might contribute to this long-term stability?*

**My answer:**

```
[Write here]
```

<details>
<summary>💡 Tip — what to aim for</summary>
- **All key neurons active**: AB/PD pacemaker active, LP and PY firing  
- **Triphasic sequence**: AB/PD → LP → PY  
- **Moderate synchrony**: R not too high, not too low  
- **Robust rhythms**: multiple weight combinations produce similar outputs  
- The circuit is stable because **gap junctions synchronise pacemaker**, and **inhibition separates phases**
</details>

---

### Final Checkpoint

* AB and PD neurons fire together because they are connected by: ______
* LP fires after AB/PD because it is first ______ and then ______
* Two different weight combinations can produce the same rhythm because: ______
* This property — many parameter combinations producing the same output — is called: ______

---

### Bonus — Think deeper

*The pyloric rhythm persists even when the STG is surgically isolated from all sensory input and from the rest of the nervous system. What does this tell you about where the rhythm is generated — is it a property of individual neurons, or of the circuit as a whole?*

**My answer:**

```
[Write here]
```
